# CPP Latent-Dynamics Pipeline — Master Notebook

This notebook is the **interactive master script** for the full analysis pipeline.  
Run cells **sequentially (top → bottom)** to reproduce every result from processed  
data through final regression outputs and publication figures.

> **TIER 4.0 compliance** — this notebook acts as the *Master Script* described  
> in the TIER Protocol.  Each section corresponds to a numbered pipeline stage  
> documented in `Scripts/pipeline_overview.md`.

| Stage | What it does | Key output |
|-------|-------------|------------|
| 0 — Bootstrap | Inject paths, verify files | — |
| S1 — Validate | Data contract check | console report |
| S2a — Train | Fit CPPForwardGRU | `Results/model_checkpoints/best_model.pt` |
| S2b — Sweep | λ grid search (optional) | `Results/sweep/` |
| S2c — Export | Full latent tensor | `Data/IntermediateData/latents_full/latents_full.npz` |
| S3 — Audit | Neural validation | `Results/validation/` |
| S4a — Ridge | RT regression | `Results/regression/` |
| S4b — Controls | Sanity checks | `Results/validation/` |
| S4c — Figures | Publication plots | `Results/validation/figures/publication_style/` |


---
## Stage 0 · Environment Bootstrap

Sets up `sys.path` so the three package directories are importable as  
`modeling`, `training`, and `analysis`.  **Always run this cell first.**


In [ ]:
import sys
from pathlib import Path

# ── Locate project root (look for conftest.py walking upward) ─────────────────
PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "conftest.py").exists():
        PROJECT_ROOT = candidate
        break

SCRIPTS = PROJECT_ROOT / "Scripts"
for _pkg in ("s1_modeling", "s2_training", "s4_analysis"):
    _p = str(SCRIPTS / _pkg)
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── Canonical paths ───────────────────────────────────────────────────────────
DATASET_DIR      = PROJECT_ROOT / "Data" / "ProcessedData"
INTERMEDIATE_DIR = PROJECT_ROOT / "Data" / "IntermediateData" / "latents_full"
RESULTS_DIR      = PROJECT_ROOT / "Results"
CHECKPOINT_PATH  = RESULTS_DIR  / "model_checkpoints" / "best_model.pt"
LATENT_PATH      = INTERMEDIATE_DIR / "latents_full.npz"

print(f"Project root  : {PROJECT_ROOT}")
print(f"Dataset dir   : {DATASET_DIR}  exists={DATASET_DIR.exists()}")
print(f"Checkpoint    : {CHECKPOINT_PATH}  exists={CHECKPOINT_PATH.exists()}")
print(f"Latents       : {LATENT_PATH}  exists={LATENT_PATH.exists()}")


### Quick dataset preview


In [ ]:
import numpy as np
import pandas as pd

eeg   = np.load(DATASET_DIR / "eeg_cpp_trials.npy")
times = np.load(DATASET_DIR / "times_ms.npy")
meta  = pd.read_csv(DATASET_DIR / "metadata.csv")

print(f"EEG shape   : {eeg.shape}  (trials × timepoints × channels)")
print(f"Time axis   : {times[0]:.0f} ms → {times[-1]:.0f} ms  ({len(times)} points)")
print(f"Metadata    : {len(meta)} rows × {len(meta.columns)} columns")
print(f"Columns     : {list(meta.columns)}")


---
## Stage S1 · Data Contract Validation

Checks that all required files exist, channel order is `(CP1, CP2, CPz)`,  
and metadata contains `trial_id` and `alignment` columns.  
Raises `ValueError` with a clear message on any failure — this is the  
**gate** before any model code runs.


In [ ]:
from modeling.data_contract import validate_stage2_dataset
from modeling.config import DataContractConfig

validate_stage2_dataset(
    dataset_dir=DATASET_DIR,
    output_dir=RESULTS_DIR / "validation",
    contract=DataContractConfig(),
)
print("✓  Data contract validation passed.")


---
## Stage S2a · Model Training

Trains `CPPForwardGRU` with the composite self-supervised CPP shape-prior loss.  
Best checkpoint is saved to `Results/model_checkpoints/best_model.pt`.

**Architecture** (`ModelConfig`):
```
Input (batch, T, 3)
  → Linear projection (3 → 16) + LayerNorm
  → Causal GRU  (hidden_dim=32, num_layers=1)
  → Reconstruction head  (hidden → 3)
  → Future-prediction head  (hidden → 3 × H, H=50 ms)
```

**Loss terms** (`LossWeights`):

| Term | λ | Purpose |
|------|---|---------|
| `lambda_recon` | 1.0 | Reconstruct current EEG frame |
| `lambda_future` | 0.2 | Predict next 50 ms |
| `lambda_derivative` | 0.5 | Match first-order slope |
| `lambda_variance` | 0.5 | Align channel variance |
| `lambda_cpp_mean` | 0.5 | CPP proxy alignment |
| `lambda_cpp_prior` | 0.1 | Shape-prior group scale |
| `lambda_smooth` | 0.001 | Latent temporal smoothness |

> ⏱ Training takes ~10–20 min on CPU (max 100 epochs, patience=15).  
> **Skip this cell** if `best_model.pt` already exists.


In [ ]:
from training.train import train_model
from modeling.config import TrainingConfig, ModelConfig, LossWeights

config = TrainingConfig(
    seed=42,
    model=ModelConfig(projection_dim=16, hidden_dim=32, num_layers=1),
    loss=LossWeights(
        lambda_recon=1.0,
        lambda_future=0.2,
        lambda_cpp_prior=0.1,
        enable_cpp_shape_prior=True,
    ),
)

if CHECKPOINT_PATH.exists():
    print(f"✓  Checkpoint already exists at:\n   {CHECKPOINT_PATH}")
    print("   Skipping training — delete the file to re-train from scratch.")
else:
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    train_model(
        dataset_dir=DATASET_DIR,
        output_dir=RESULTS_DIR / "model_checkpoints",
        config=config,
    )
    print("✓  Training complete.")


---
## Stage S2b · Hyperparameter Sweep *(optional)*

Scans `lambda_cpp_prior` ∈ {0.0, 0.05, 0.1, 0.2, 0.5} with 3 seeds each.  
Outputs per-config validation loss summaries and a Markdown best-config report.

> ℹ️  Production model used `lambda_cpp_prior = 0.1` (config `long_002`).  
> This cell is commented out — uncomment only if you need to re-run the sweep.


In [ ]:
# from training.sweep import run_cpp_prior_sweep
# from modeling.config import TrainingConfig
#
# run_cpp_prior_sweep(
#     dataset_dir=DATASET_DIR,
#     output_dir=RESULTS_DIR / "sweep",
#     base_config=TrainingConfig(),
# )
# print("✓  Sweep complete.")

print("(Sweep skipped — using committed best config: lambda_cpp_prior = 0.1)")


---
## Stage S2c · Export Full Latent States

Runs the **frozen** best model in `eval()` mode over all 7 297 trials and  
saves the complete hidden-state tensor.

**Output:** `Data/IntermediateData/latents_full/latents_full.npz`  
- `latents` : `(7297, 308, 32)` — hidden state at every time point  
- `times_ms` : `(308,)` — shared time axis  
- `trial_ids` : `(7297,)` — row-aligned trial identifiers  

This `.npz` file is the **input to all downstream analyses (S3, S4)**.


In [ ]:
from training.train import export_full_latents_from_checkpoint

if LATENT_PATH.exists():
    z = np.load(LATENT_PATH)
    print(f"✓  Latents already exist — shape: {z['latents'].shape}")
else:
    INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
    export_full_latents_from_checkpoint(
        checkpoint_path=CHECKPOINT_PATH,
        dataset_dir=DATASET_DIR,
        output_dir=INTERMEDIATE_DIR,
    )
    print("✓  Latents exported.")


---
## Stage S3 · Neural Validation Audit

Strict audit of what information the hidden states encode:

| Decoding target | Expected result | Interpretation |
|----------------|-----------------|----------------|
| CPP amplitude (R²) | ~0.97 | Hidden states strongly encode CPP |
| CPP amplitude (Δ R² vs baseline) | ~0.95 | Increment is genuine |
| RT bin (fast/slow tertile) | marginal > chance | Weak but real RT signal |
| Choice direction | ~chance | Not encoding stimulus identity ✓ |
| Experimental condition | ~chance | Not encoding task context ✓ |

Outputs: `Results/validation/hidden_state_classification_decoding.csv`,  
`Results/validation/hidden_state_neural_regression_decoding.csv`,  
`validation_summary.md`, and all diagnostic figures.


In [ ]:
import subprocess, sys as _sys

result = subprocess.run(
    [_sys.executable,
     str(SCRIPTS / "s3_validation" / "run_neural_validation_audit.py"),
     "--latent-path",  str(LATENT_PATH),
     "--dataset-dir",  str(DATASET_DIR),
     "--output-dir",   str(RESULTS_DIR / "validation")],
    capture_output=True, text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Neural validation audit failed.")
print("✓  Neural validation audit complete.")


---
## Stage S4a · Ridge Regression — Hidden States → RT

Predicts `log(RT_ms)` from window-averaged hidden states using Ridge regression  
with nested 5-fold cross-validation.

**Four pre-response windows tested:**

| Window | Expected Δ R² (hidden vs baseline) |
|--------|-------------------------------------|
| −600 to −300 ms | **+0.103** (strongest) |
| −300 to −120 ms | moderate |
| −120 to −50 ms  | moderate |
| −600 to −50 ms  | broadband |

Shuffled-hidden control R² ≈ 0.195 confirms the increment is not overfitting.


In [ ]:
from analysis.rt_ridge import run_ridge_rt_analysis

run_ridge_rt_analysis(
    latent_npz=LATENT_PATH,
    dataset_dir=DATASET_DIR,
    output_dir=RESULTS_DIR / "regression",
)
print("✓  Ridge RT analysis complete.")

# Show the main summary table if it was written
summary_csv = RESULTS_DIR / "regression" / "ridge_rt_performance.csv"
if summary_csv.exists():
    print()
    print(pd.read_csv(summary_csv).to_string(index=False))


---
## Stage S4b · Minimal Sanity Controls

Verifies that training actually matters by comparing the trained model  
reconstruction against:

1. **Untrained baseline** — random GRU weights, same architecture  
2. **Time-shuffled latent** — same hidden states but shuffled along the time axis  

Both baselines should show substantially worse CPP reconstruction  
(lower R², higher loss).


In [ ]:
from training.controls import run_minimal_controls
from modeling.config import TrainingConfig

run_minimal_controls(
    dataset_dir=DATASET_DIR,
    output_dir=RESULTS_DIR / "validation" / "controls",
    config=TrainingConfig(),
)
print("✓  Sanity controls complete.")


---
## Stage S4c · Publication Figures

Assembles all pre-computed results into publication-ready figures.  
**Run last**, after all upstream outputs are in place.

- **Figure 2** — hidden-state vs CPP relationship (scatter + time-course + decoding bars)
- **Supplementary S1** — behavioural external validation (fast-condition RT)


In [ ]:
result = subprocess.run(
    [_sys.executable,
     str(RESULTS_DIR / "validation" / "make_publication_figures.py")],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("Figure generation failed.")
print("✓  Publication figures generated.")


---
## ✅ Pipeline Complete

| Stage | Output path |
|-------|------------|
| S1 validation | `Results/validation/validation_summary.md` |
| S2a model | `Results/model_checkpoints/best_model.pt` |
| S2c latents | `Data/IntermediateData/latents_full/latents_full.npz` |
| S3 audit | `Results/validation/hidden_state_classification_decoding.csv` |
| S4a regression | `Results/regression/ridge_rt_performance.csv` |
| S4b controls | `Results/validation/` |
| S4c figures | `Results/validation/figures/publication_style/` |

---
For a full narrative of each step, see `Scripts/pipeline_overview.md`.  
To run the test suite: `pytest` from the project root.
